# FLUKE Coreference Resolution with GPT-5

This notebook evaluates coreference resolution robustness using OpenAI's GPT-5 model with FLUKE linguistic modifications.

In [1]:
# Standard imports
from datasets import load_dataset
import dspy
import openai
import os
import pandas as pd
import json
import glob
import time
import random
from dotenv import load_dotenv
from dspy.evaluate import Evaluate

# Import unified FLUKE utilities
from fluke_reasoning_utils import (
    REASONING_MODELS, REASONING_CONFIGS,
    remove_space, extract_classification_prediction,
    aggregate_results, highlight_drops_and_significance,
    compare_models
)

/Users/hungthinh/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables
load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

# Select GPT-5 configuration
CONFIG_NAME = 'standard'  # Options: 'standard', 'detailed', 'turbo'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Description: {config['description']}")

# Configure DSPy with GPT-5
# GPT-5 only supports temperature=1
lm = dspy.LM(MODEL_ID, max_tokens=250)
dspy.configure(lm=lm)

In [3]:
# Select GPT-5 configuration
CONFIG_NAME = 'standard'  # Options: 'standard', 'detailed', 'turbo'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Description: {config['description']}")

# Configure DSPy with GPT-5
lm = dspy.LM(MODEL_ID, temperature=1, max_tokens=20_000)
dspy.configure(lm=lm)

Configuration: standard
Model: gpt-5 (openai/gpt-5)
Description: Standard reasoning approach with GPT-5


## Load Coreference Data

In [4]:
# Load coreference dataset
ds = pd.read_json('../../../data/train_dev_test_data/coref/test.json')
ds = ds.to_dict('records')

print(f"Loaded {len(ds)} coreference samples")

# Split by label
positive_samples = []
negative_samples = []
for i, x in enumerate(ds):
    if x["label"] == 1:
        positive_samples.append((i, x))
    else:
        negative_samples.append((i, x))

print(f"Positive samples (coreferent): {len(positive_samples)}")
print(f"Negative samples (not coreferent): {len(negative_samples)}")

# Combine and shuffle
samples = positive_samples + negative_samples
random.shuffle(samples)

Loaded 1517 coreference samples
Positive samples (coreferent): 500
Negative samples (not coreferent): 1017


In [16]:
# Create examples
examples = [
    dspy.Example({
        "text": remove_space(r["text"]),
        "pronoun": r["pronoun"],
        "candidates": '0: ' + r["candidates"][0] + ', 1: ' + r["candidates"][1],
        "label": r['label']
    }).with_inputs("text", "pronoun", "candidates")
    for i, r in samples
]

# Test example
example = examples[0]
print(f"\nExample text: {example.text}")
print(f"Pronoun: {example.pronoun}")
print(f"Candidates: {example.candidates}")
print(f"Label: {example.label}")


Example text: Maira meanwhile is angry with Adhira so Bari Sarkar consoles her.
Pronoun: her
Candidates: 0: Maira, 1: Adhira
Label: 0


## Define Task with GPT-5

In [18]:
class GPT5Coref(dspy.Signature):
    """Determine the candidate that the given pronoun refers to in the text. Answer with the index of the candidate."""
    text = dspy.InputField()
    pronoun = dspy.InputField()
    candidates = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class GPT5CorefModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(GPT5Coref)

    def forward(self, text, pronoun, candidates):
        return self.prog(text=text, pronoun=pronoun, candidates=candidates)

# Initialize module
gpt5_coref = GPT5CorefModule()

# Evaluation metric
def eval_metric(true, prediction, trace=None):
    pred = prediction.label
    parsed_answer = extract_classification_prediction(pred)
    return parsed_answer == str(true.label)

In [19]:
# Test single example
pred = gpt5_coref(text=example.text, pronoun=example.pronoun, candidates=example.candidates)
print(f"Text: {example.text}")
print(f"Pronoun: {example.pronoun}")
print(f"Candidates: {example.candidates}")
print(f"True Label: {example.label}")
print(f"Prediction: {pred.label}")
print(f"Correct: {eval_metric(example, pred)}")

Text: Maira meanwhile is angry with Adhira so Bari Sarkar consoles her.
Pronoun: her
Candidates: 0: Maira, 1: Adhira
True Label: 0
Prediction: 0
Correct: True


## Evaluate Original Dataset

In [25]:
# GPT-5 can handle larger batches
TEST_SIZE = 200  # Can increase for GPT-5
test_examples = examples

print(f"Evaluating {len(test_examples)} examples with GPT-5...")

evaluate = Evaluate(
    devset=test_examples,
    metric=eval_metric,
    num_threads=4,  # GPT-5 can handle more threads
    display_progress=True,
    display_table=10,
    return_all_scores=True
)

results = evaluate(gpt5_coref)

# Save results
print(results)
items = []
for sample in results['results']:
    print(sample)
    items.append({
        'text': sample[0]['text'],
        'pronoun': sample[0]['pronoun'],
        'candidates': str(sample[0]['candidates']),
        'label': sample[0]['label'],
        'pred': extract_classification_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']
    })

df_result = pd.DataFrame(items)
output_file = f'../results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv'
df_result.to_csv(output_file, index=False)

print(f"\nGPT-5 Accuracy: {results['score']:.3f}")
print(f"Results saved to: {output_file}")

Evaluating 1517 examples with GPT-5...
Average Metric: 1258.00 / 1517 (82.9%): 100%|██████████| 1517/1517 [00:03<00:00, 482.77it/s] 

2025/08/17 12:20:56 INFO dspy.evaluate.evaluate: Average Metric: 1258 / 1517 (82.9%)


,text,pronoun,candidates,example_label,pred_label,eval_metric
0,Maira meanwhile is angry with Adhira so Bari Sarkar consoles her.,her,"0: Maira, 1: Adhira",0,0,✔️ [True]
1,"Esaias was not related to Willem van de Velde, but he was the cous...",he,"0: Esaias, 1: Willem van de Velde",0,0,✔️ [True]
2,Ed Helms was cast as Derek Smeathe but scheduling conflicts preven...,him,"0: Derek Smeathe, 1: Ed Helms",1,1,✔️ [True]
3,"Indi asks Mink to stay, but she departs, leaving a note for Romeo.",she,"0: Indi, 1: Mink",0,1,
4,"Layla initially thinks Carolyn is stuck up, especially when she di...",she,"0: Layla, 1: Carolyn",1,1,✔️ [True]
5,"While searching, nurse Jill is killed upon encountering Michaela, ...",she,"0: nurse Jill, 1: Michaela",0,0,✔️ [True]
6,The spear easily pierced through the armor because it was too thin.,it,"0: The spear, 1: the armor",1,1,✔️ [True]
7,Anna Dalassene consented to this but forced Katherine to resign im...,she,"0: Katherine, 1: Anna Dalassene",0,0,✔️ [True]
8,Arushi does not listen to Sunaina when she tells him that she woul...,she,"0: Arushi, 1: Sunaina",1,1,✔️ [True]
9,"David truly loved and trusted Wolfgang now, where before he had be...",he,"0: David, 1: Wolfgang",0,0,✔️ [True]


EvaluationResult(score=82.93, results=<list of 1517 results>)
(Example({'text': 'Maira meanwhile is angry with Adhira so Bari Sarkar consoles her.', 'pronoun': 'her', 'candidates': '0: Maira, 1: Adhira', 'label': 0}) (input_keys={'text', 'pronoun', 'candidates'}), Prediction(
    label='0'
), True)
(Example({'text': 'Esaias was not related to Willem van de Velde, but he was the cousin of Jan van de Velde.', 'pronoun': 'he', 'candidates': '0: Esaias, 1: Willem van de Velde', 'label': 0}) (input_keys={'text', 'pronoun', 'candidates'}), Prediction(
    label='0'
), True)
(Example({'text': 'Ed Helms was cast as Derek Smeathe but scheduling conflicts prevented him from taking the role.', 'pronoun': 'him', 'candidates': '0: Derek Smeathe, 1: Ed Helms', 'label': 1}) (input_keys={'text', 'pronoun', 'candidates'}), Prediction(
    label='1'
), True)
(Example({'text': 'Indi asks Mink to stay, but she departs, leaving a note for Romeo.', 'pronoun': 'she', 'candidates': '0: Indi, 1: Mink', 'label'

## Evaluate Modifications

In [40]:
def evaluate_modified_set(data, program, max_samples=50):
    """Evaluate on modified dataset with GPT-5."""
    limited_data = data[:max_samples] if len(data) > max_samples else data
    
    mod_examples = [
        dspy.Example({
            "text": remove_space(r['modified_text']),
            "original_text": remove_space(r['original_text']),
            "pronoun": r['modified_pronoun'],
            "candidates": '0: ' + r["modified_candidates"][0] + ', 1: ' + r["modified_candidates"][1],
            "label": int(r.get('modified_label') if 'modified_label' in r else r['label']),
            "original_label": int(r['original_label'] if 'original_label' in r else r['label']),
            "type": r['type'] if 'type' in r else None,
            "id": r['index']
        }).with_inputs("text", "pronoun", "candidates")
        for r in limited_data
    ]
    
    evaluate = Evaluate(
        devset=mod_examples,
        metric=eval_metric,
        num_threads=4,  # GPT-5 can handle more threads
        display_progress=True,
        display_table=1,
        return_all_scores=True
    )
    
    return evaluate(program)

In [42]:
# Load original predictions
original_pred_file = f'../results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file)
    original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
    print(f"Loaded original GPT-5 predictions from {original_pred_file}")
else:
    print("Please run original evaluation first")
    original_pred_ds = None

# Test all modifications with GPT-5
json_files = glob.glob('../../../data/modified_data/coref/*_100.json')
# GPT-5 can handle more modifications

print(f"\nTesting {len(json_files)} modifications with GPT-5...")

for json_file in json_files:
    print(f"\nProcessing: {json_file.split('/')[-1]}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    print(data[0])
    # GPT-5 can handle larger samples
    results_mod = evaluate_modified_set(data, gpt5_coref, max_samples=150)
    
    # Process results
    items = []
    for sample in results_mod['results']:
        item = {
            'text': sample[0]['text'],
            'original_text': sample[0]['original_text'],
            'pronoun': sample[0]['pronoun'],
            'candidates': str(sample[0]['candidates']),
            'type': sample[0]['type'],
            'modified_label': sample[0]['label'],
            'original_label': sample[0]['original_label'],
            'modified_pred': extract_classification_prediction(sample[1]['label']),
            'raw_output': sample[1]['label'],
            'id': sample[0]['id']
        }
        
        # Find original prediction
        if original_pred_ds is not None:
            matches = original_pred_ds[original_pred_ds.index == item['id']]
            item['original_pred'] = matches.iloc[0]['pred'] if not matches.empty else None
        else:
            item['original_pred'] = None
        
        items.append(item)
    
    df_mod = pd.DataFrame(items)
    mod_name = json_file.split('/')[-1].replace('.json', '')
    output_file = f'../results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-{mod_name}.csv'
    df_mod.to_csv(output_file, index=False)
    
    print(f"Accuracy: {results_mod['score']:.3f}")
    print(f"Saved to: {output_file}")
    
    time.sleep(2)  # Shorter delay for GPT-5

Loaded original GPT-5 predictions from ../results/coref/gpt-5-standard-0shot-coref.csv

Testing 18 modifications with GPT-5...

Processing: casual_100.json
{'original_text': 'Sally becomes attracted to Bella, but she is disappointed to learn she is marrying Emily Vincent.', 'original_candidates': ['Sally', 'Bella'], 'original_pronoun': 'she', 'original_label': 1, 'modified_text': "Sally's got a crush on Bella, but she's bummed out to find out she's tying the knot with Emily Vincent.", 'type': 'casual', 'modified_candidates': ['Sally', 'Bella'], 'modified_pronoun': 'she', 'modified_label': 1, 'index': 1084}
Average Metric: 87.00 / 100 (87.0%): 100%|██████████| 100/100 [03:05<00:00,  1.86s/it]


2025/08/19 11:47:17 INFO dspy.evaluate.evaluate: Average Metric: 87 / 100 (87.0%)


,text,original_text,pronoun,candidates,example_label,original_label,type,id,pred_label,eval_metric
0,"Sally's got a crush on Bella, but she's bummed out to find out she...","Sally becomes attracted to Bella, but she is disappointed to learn...",she,"0: Sally, 1: Bella",1,1,casual,1084,0,


Accuracy: 87.000
Saved to: ../results/coref/gpt-5-standard-0shot-casual_100.csv

Processing: discourse_100.json
{'original_text': 'The villagers attempt to have the werewolves executed, but they are instead exiled by the clan leader.', 'original_candidates': ['the werewolves', 'The villagers'], 'original_pronoun': 'they', 'original_label': 0, 'modified_text': 'The villagers attempt to have the werewolves executed, however, they are instead exiled by the clan leader.', 'type': 'reverse', 'modified_candidates': ['the werewolves', 'The villagers'], 'modified_pronoun': 'they', 'modified_label': 0, 'index': 842}
Average Metric: 84.00 / 100 (84.0%): 100%|██████████| 100/100 [00:20<00:00,  4.91it/s]

2025/08/19 11:47:44 INFO dspy.evaluate.evaluate: Average Metric: 84 / 100 (84.0%)


...

,text,original_text,pronoun,candidates,example_label,original_label,type,id,pred_label,eval_metric
0,"The delivery lorry zoom past the school bus lor, cos it was going ...",The delivery truck zoomed by the school bus because it was going s...,it,"0: 0, 1: :",0,0,singaporean_english,1284,0,✔️ [True]


Accuracy: 74.510
Saved to: ../results/coref/gpt-5-standard-0shot-singlish_100.csv

Processing: negation_100.json
{'original_text': 'Kathy paid Jane to leave but she returned several weeks later.', 'original_candidates': ['Kathy', 'Jane'], 'original_pronoun': 'she', 'original_label': 1, 'modified_text': 'Kathy paid Jane to leave but she did not return several weeks later.', 'type': 'verbal', 'modified_candidates': ['Kathy', 'Jane'], 'modified_pronoun': 'she', 'modified_label': 1, 'test': 'negation', 'index': 991}
Average Metric: 72.00 / 98 (73.5%): 100%|██████████| 98/98 [00:01<00:00, 62.24it/s]

2025/08/19 11:49:25 INFO dspy.evaluate.evaluate: Average Metric: 72 / 98 (73.5%)


,text,original_text,pronoun,candidates,example_label,original_label,type,id,pred_label,eval_metric
0,Kathy paid Jane to leave but she did not return several weeks later.,Kathy paid Jane to leave but she returned several weeks later.,she,"0: Kathy, 1: Jane",1,1,verbal,991,1,✔️ [True]


Accuracy: 73.470
Saved to: ../results/coref/gpt-5-standard-0shot-negation_100.csv


## Chain-of-Thought with GPT-5

In [ ]:
class CoTGPT5Coref(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(GPT5Coref)

    def forward(self, text, pronoun, candidates):
        return self.prog(text=text, pronoun=pronoun, candidates=candidates)

# Test CoT
cot_gpt5_coref = CoTGPT5Coref()
pred_cot = cot_gpt5_coref(text=example.text, pronoun=example.pronoun, candidates=example.candidates)
print("Chain-of-Thought with GPT-5:")
print(f"Text: {example.text}")
print(f"Pronoun: {example.pronoun}")
print(f"Candidates: {example.candidates}")
print(f"\nReasoning: {pred_cot.reasoning if hasattr(pred_cot, 'reasoning') else 'N/A'}")
print(f"\nPrediction: {pred_cot.label}")

## Aggregate Results

In [ ]:
# Aggregate all modification results
result_files = glob.glob(f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')

if result_files:
    results_df = aggregate_results(
        result_files,
        task_name='coreference_resolution',
        model_name=f'{MODEL_NAME}-{CONFIG_NAME}'
    )
    
    if not results_df.empty:
        # Display summary
        print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
        print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])
        
        # Save aggregated results
        output_file = f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-DP.csv'
        results_df.to_csv(output_file, index=False)
        print(f"\nAggregated results saved to: {output_file}")
        
        # Display styled results
        styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
        display(styled_df)
else:
    print("No result files found to aggregate")

## Model Comparison

In [ ]:
# Compare GPT-5 with other models
comparison_files = {
    'GPT-5': f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv',
    'GPT-4o': 'results/coref/gpt4o-0shot-coref.csv',
    'Claude-3.5': 'results/coref/claude-3-5-sonnet-0shot-coref.csv',
    'o3-2025-04-16': 'results/coref/o3-2025-04-16-standard-0shot-coref.csv',
    'Mixtral-8x22B': 'results/coref/mixtral-8x22b-0shot-coref.csv'
}

comparison_df = compare_models(comparison_files, task_name='coreference_resolution')

if not comparison_df.empty:
    print("\nModel Comparison (including GPT-5):")
    print(comparison_df)
    
    # Calculate GPT-5 improvement
    if 'GPT-5' in comparison_df['Model'].values:
        gpt5_acc = comparison_df[comparison_df['Model'] == 'GPT-5']['Accuracy'].values[0]
        other_accs = comparison_df[comparison_df['Model'] != 'GPT-5']['Accuracy'].values
        if len(other_accs) > 0:
            avg_others = other_accs.mean()
            improvement = gpt5_acc - avg_others
            print(f"\nGPT-5 Performance: {gpt5_acc:.3f}")
            print(f"Average of other models: {avg_others:.3f}")
            print(f"GPT-5 improvement: {improvement:+.3f} ({improvement*100:+.1f}%)")
    
    # Highlight best performer
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]
    
    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No comparison data available")

## GPT-5 Performance Analysis

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Coreference Resolution with GPT-5 Complete!")
print(f"{'='*60}")

if 'results' in locals():
    print(f"\nBase accuracy: {results[0]:.3f}")

if 'results_df' in locals() and not results_df.empty:
    avg_row = results_df[results_df['modification'] == 'average'].iloc[0]
    print(f"Average robustness drop: {avg_row['difference']:.3f}")
    print(f"Modifications tested: {len(results_df) - 1}")

print(f"\nGPT-5 Configuration: {config['description']}")
print(f"\nKey advantages of GPT-5:")
print("• Advanced pronoun resolution capabilities")
print("• Better understanding of grammatical agreement")
print("• Improved semantic plausibility reasoning")
print("• Faster inference than o3 models")
print("• Higher throughput with multi-threading")

print(f"\nFiles saved in: results/coref/")